# Ollama Portfolio Analyzer

This notebook combines deterministic portfolio reconstruction, live price data from `yfinance`, benchmark data from Yahoo Finance using `^GSPC`, and local narrative analysis through Ollama.


In [177]:
# If needed, run this once in the notebook environment:
# %pip install pandas yfinance


In [178]:
from __future__ import annotations

import json
import math
import re
import subprocess
import urllib.error
import urllib.request
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import yfinance as yf


In [179]:
MODEL_NAME = "llama3.1:latest"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
NUM_PREDICT = 4000
TEMPERATURE = 0.4

CSV_PATH = Path("../data/raw/mantis_invest.csv")
RISK_PROFILE = 65
BENCHMARK_SYMBOL = "^GSPC"
PROJECTION_YEARS = 18


In [180]:
@dataclass
class PositionLot:
    quantity: float
    unit_cost: float
    buy_date: datetime


def parse_money(value: Any) -> float:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return 0.0
    text = str(value).strip().replace("$", "").replace(",", "")
    if not text:
        return 0.0
    negative = text.startswith("(") and text.endswith(")")
    if negative:
        text = text[1:-1]
    try:
        amount = float(text)
    except ValueError:
        return 0.0
    return -amount if negative else amount


def parse_quantity(value: Any) -> float:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return 0.0
    text = str(value).strip().replace(",", "")
    if not text:
        return 0.0
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else 0.0


def load_transactions(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df["Activity Date"] = pd.to_datetime(df["Activity Date"], errors="coerce")
    df = df.dropna(subset=["Activity Date"]).copy()
    df["Amount_num"] = df.get("Amount", 0).apply(parse_money)
    df["Price_num"] = df.get("Price", 0).apply(parse_money)
    df["Quantity_num"] = df.get("Quantity", 0).apply(parse_quantity)
    df["Instrument"] = df["Instrument"].fillna("").astype(str).str.strip()
    df["Trans Code"] = df["Trans Code"].fillna("").astype(str).str.strip()
    return df.sort_values("Activity Date").reset_index(drop=True)


def holding_period_bucket(days: float) -> str:
    if days <= 30:
        return "<=30d"
    if days < 365:
        return "31-364d"
    return ">=365d"


In [181]:
def weighted_average_date(dates: list[datetime], weights: list[float]) -> str | None:
    if not dates or not weights or sum(weights) <= 0:
        return None
    weighted_ordinal = sum(date.toordinal() * weight for date, weight in zip(dates, weights)) / sum(weights)
    return datetime.fromordinal(int(round(weighted_ordinal))).date().isoformat()


def build_lot_analytics(df: pd.DataFrame) -> dict[str, Any]:
    lots: dict[str, deque[PositionLot]] = defaultdict(deque)
    realized_pnl: dict[str, float] = defaultdict(float)
    closed_lots: list[dict[str, Any]] = []
    share_credit_codes = {"REC"}

    for _, row in df.iterrows():
        symbol = row["Instrument"]
        code = row["Trans Code"]
        if not symbol:
            continue

        trade_date = row["Activity Date"].to_pydatetime()
        quantity = float(row["Quantity_num"])
        amount = float(row["Amount_num"])

        if code == "Buy" and quantity > 0:
            lots[symbol].append(
                PositionLot(
                    quantity=quantity,
                    unit_cost=(-amount / quantity) if quantity else 0.0,
                    buy_date=trade_date,
                )
            )
        elif code == "SPL" and quantity > 0:
            total_qty = sum(lot.quantity for lot in lots[symbol])
            if total_qty > 0:
                factor = (total_qty + quantity) / total_qty
                adjusted = deque()
                for lot in lots[symbol]:
                    adjusted.append(
                        PositionLot(
                            quantity=lot.quantity * factor,
                            unit_cost=lot.unit_cost / factor,
                            buy_date=lot.buy_date,
                        )
                    )
                lots[symbol] = adjusted
        elif code in share_credit_codes and quantity > 0:
            lots[symbol].append(PositionLot(quantity=quantity, unit_cost=0.0, buy_date=trade_date))
        elif code == "Sell" and quantity > 0:
            remaining = quantity
            sell_unit_price = amount / quantity if quantity else 0.0

            while remaining > 1e-9 and lots[symbol]:
                lot = lots[symbol][0]
                take = min(remaining, lot.quantity)
                cost_basis = take * lot.unit_cost
                proceeds = take * sell_unit_price
                pnl = proceeds - cost_basis
                realized_pnl[symbol] += pnl
                closed_lots.append(
                    {
                        "ticker": symbol,
                        "quantity": round(take, 6),
                        "buy_date": lot.buy_date.date().isoformat(),
                        "sell_date": trade_date.date().isoformat(),
                        "hold_days": (trade_date - lot.buy_date).days,
                        "unit_cost": round(lot.unit_cost, 6),
                        "sell_unit_price": round(sell_unit_price, 6),
                        "cost_basis": round(cost_basis, 2),
                        "proceeds": round(proceeds, 2),
                        "realized_pnl": round(pnl, 2),
                    }
                )
                lot.quantity -= take
                remaining -= take
                if lot.quantity <= 1e-9:
                    lots[symbol].popleft()

    current_positions = []
    for symbol, queue in lots.items():
        total_qty = sum(lot.quantity for lot in queue)
        if total_qty <= 1e-9:
            continue
        total_cost = sum(lot.quantity * lot.unit_cost for lot in queue)
        buy_dates = [lot.buy_date for lot in queue]
        weights = [lot.quantity for lot in queue]
        current_positions.append(
            {
                "ticker": symbol,
                "quantity": round(total_qty, 6),
                "cost_basis_total": round(total_cost, 2),
                "avg_cost": round(total_cost / total_qty, 6) if total_qty else 0.0,
                "first_buy_date": min(buy_dates).date().isoformat(),
                "weighted_avg_buy_date": weighted_average_date(buy_dates, weights),
            }
        )

    current_positions.sort(key=lambda item: item["cost_basis_total"], reverse=True)

    return {
        "current_positions": current_positions,
        "closed_lots": closed_lots,
        "realized_pnl_by_ticker": {
            ticker: round(float(pnl), 2)
            for ticker, pnl in sorted(realized_pnl.items(), key=lambda item: item[1], reverse=True)
        },
    }


def summarize_portfolio(df: pd.DataFrame, lot_data: dict[str, Any]) -> dict[str, Any]:
    buy_df = df[df["Trans Code"] == "Buy"]
    sell_df = df[df["Trans Code"] == "Sell"]
    ach_df = df[df["Trans Code"] == "ACH"]
    dividend_df = df[df["Trans Code"] == "CDIV"]
    interest_df = df[df["Trans Code"] == "INT"]

    holding_periods = [lot["hold_days"] for lot in lot_data["closed_lots"]]
    holding_counter = Counter(holding_period_bucket(days) for days in holding_periods)
    years = max(((df["Activity Date"].max() - df["Activity Date"].min()).days / 365.25), 0.01)

    buys_by_ticker = buy_df.groupby("Instrument")["Amount_num"].sum().abs().sort_values(ascending=False)
    sells_by_ticker = sell_df.groupby("Instrument")["Amount_num"].sum().sort_values(ascending=False)

    return {
        "date_range": {
            "start": df["Activity Date"].min().date().isoformat(),
            "end": df["Activity Date"].max().date().isoformat(),
            "years": round(years, 2),
        },
        "transaction_counts": {
            "rows": int(len(df)),
            "buys": int(len(buy_df)),
            "sells": int(len(sell_df)),
            "distinct_symbols_traded": int(df.loc[df["Instrument"].ne(""), "Instrument"].nunique()),
        },
        "cash_flows": {
            "net_deposits": round(float(ach_df["Amount_num"].sum()), 2),
            "gross_buys": round(float(buy_df["Amount_num"].sum() * -1), 2),
            "gross_sells": round(float(sell_df["Amount_num"].sum()), 2),
            "dividends": round(float(dividend_df["Amount_num"].sum()), 2),
            "interest": round(float(interest_df["Amount_num"].sum()), 2),
            "cash_balance_estimate": round(float(df["Amount_num"].sum()), 2),
        },
        "behavioral_metrics": {
            "sell_to_buy_ratio": round(float(sell_df["Amount_num"].sum()) / max(float(buy_df["Amount_num"].sum() * -1), 1.0), 3),
            "annualized_turnover_proxy": round((float(sell_df["Amount_num"].sum()) / max(float(buy_df["Amount_num"].sum() * -1), 1.0)) / years, 3),
            "holding_period_buckets": dict(holding_counter),
            "median_holding_period_days": round(float(pd.Series(holding_periods).median()), 1) if holding_periods else None,
        },
        "current_positions": lot_data["current_positions"],
        "current_positions_top_15": lot_data["current_positions"][:15],
        "top_deployed_capital": [
            {
                "ticker": ticker,
                "gross_buy_amount": round(float(amount * -1), 2),
                "gross_sell_amount": round(float(sells_by_ticker.get(ticker, 0.0)), 2),
                "realized_pnl": round(float(lot_data["realized_pnl_by_ticker"].get(ticker, 0.0)), 2),
            }
            for ticker, amount in buys_by_ticker.head(15).items()
        ],
        "realized_pnl_by_ticker": [
            {"ticker": ticker, "realized_pnl": pnl}
            for ticker, pnl in lot_data["realized_pnl_by_ticker"].items()
        ],
    }


In [182]:
def extract_close_frame(downloaded: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    if downloaded.empty:
        return pd.DataFrame(columns=tickers)

    if isinstance(downloaded.columns, pd.MultiIndex):
        if "Close" in downloaded.columns.get_level_values(0):
            close = downloaded["Close"].copy()
        else:
            close = downloaded.xs("Close", axis=1, level=-1).copy()
    else:
        close = downloaded[["Close"]].copy()
        close.columns = tickers[:1]

    if isinstance(close, pd.Series):
        close = close.to_frame(name=tickers[0])

    close = close.apply(pd.to_numeric, errors="coerce")
    close.index = pd.to_datetime(close.index)
    return close.sort_index()


def fetch_market_data(traded_symbols: list[str], benchmark_symbol: str, start_date: str) -> dict[str, Any]:
    tickers = sorted(set(symbol for symbol in traded_symbols if symbol))
    history_start = (pd.Timestamp(start_date) - pd.Timedelta(days=10)).strftime("%Y-%m-%d")

    ticker_history_raw = yf.download(tickers=tickers, start=history_start, auto_adjust=False, progress=False, threads=True)
    ticker_close = extract_close_frame(ticker_history_raw, tickers)

    benchmark_raw = yf.download(tickers=benchmark_symbol, start=history_start, auto_adjust=False, progress=False, threads=False)
    benchmark_close = extract_close_frame(benchmark_raw, [benchmark_symbol]).iloc[:, 0].dropna()
    benchmark_close.name = benchmark_symbol

    latest_prices = ticker_close.ffill().iloc[-1].dropna().to_dict() if not ticker_close.empty else {}
    latest_benchmark_price = float(benchmark_close.iloc[-1]) if not benchmark_close.empty else None

    return {
        "ticker_close": ticker_close,
        "latest_prices": latest_prices,
        "benchmark_close": benchmark_close,
        "latest_benchmark_price": latest_benchmark_price,
    }


def next_market_date(index: pd.Index, dt: Any) -> pd.Timestamp:
    ts = pd.Timestamp(dt).normalize()
    pos = index.searchsorted(ts)
    if pos >= len(index):
        pos = len(index) - 1
    return index[pos]


def price_return_from_date(price_series: pd.Series, start_date: Any) -> float | None:
    if price_series.empty:
        return None
    start_ts = next_market_date(price_series.index, start_date)
    start_price = float(price_series.loc[start_ts])
    end_price = float(price_series.iloc[-1])
    if start_price <= 0:
        return None
    return (end_price / start_price) - 1


def price_return_between_dates(price_series: pd.Series, start_date: Any, end_date: Any) -> float | None:
    if price_series.empty:
        return None
    start_ts = next_market_date(price_series.index, start_date)
    end_ts = next_market_date(price_series.index, end_date)
    start_price = float(price_series.loc[start_ts])
    end_price = float(price_series.loc[end_ts])
    if start_price <= 0:
        return None
    return (end_price / start_price) - 1


In [183]:
def build_daily_share_matrix(df: pd.DataFrame, market_index: pd.Index, tickers: list[str]) -> pd.DataFrame:
    share_credit_codes = {"REC"}
    events: list[dict[str, Any]] = []

    for _, row in df.iterrows():
        symbol = row["Instrument"]
        code = row["Trans Code"]
        qty = float(row["Quantity_num"])
        if not symbol or symbol not in tickers or qty <= 0:
            continue

        delta = 0.0
        if code == "Buy":
            delta = qty
        elif code == "Sell":
            delta = -qty
        elif code == "SPL":
            delta = qty
        elif code in share_credit_codes:
            delta = qty

        if delta == 0.0:
            continue

        events.append({
            "date": next_market_date(market_index, row["Activity Date"]),
            "ticker": symbol,
            "delta": delta,
        })

    if not events:
        return pd.DataFrame(index=market_index, columns=tickers).fillna(0.0)

    event_df = pd.DataFrame(events)
    share_changes = event_df.pivot_table(index="date", columns="ticker", values="delta", aggfunc="sum")
    share_changes = share_changes.reindex(market_index, fill_value=0.0)
    share_matrix = share_changes.cumsum()
    return share_matrix.reindex(columns=tickers, fill_value=0.0)


def build_cash_balance_series(df: pd.DataFrame, market_index: pd.Index) -> pd.Series:
    cash_events = [
        {"date": next_market_date(market_index, row["Activity Date"]), "amount": float(row["Amount_num"])}
        for _, row in df.iterrows()
    ]
    cash_df = pd.DataFrame(cash_events)
    cash_changes = cash_df.groupby("date")["amount"].sum().reindex(market_index, fill_value=0.0)
    return cash_changes.cumsum()


def build_trade_matched_benchmark_series(df: pd.DataFrame, benchmark_close: pd.Series) -> tuple[pd.Series, float]:
    trade_flows = df[df["Trans Code"].isin(["Buy", "Sell"])].copy()
    share_events = pd.Series(0.0, index=benchmark_close.index)

    for _, row in trade_flows.iterrows():
        amount = float(row["Amount_num"])
        if amount == 0:
            continue
        trade_date = next_market_date(benchmark_close.index, row["Activity Date"])
        benchmark_price = float(benchmark_close.loc[trade_date])
        if benchmark_price <= 0:
            continue

        if row["Trans Code"] == "Buy":
            share_events.loc[trade_date] += (-amount) / benchmark_price
        elif row["Trans Code"] == "Sell":
            share_events.loc[trade_date] -= amount / benchmark_price

    shares = share_events.cumsum()
    benchmark_value_series = shares * benchmark_close
    invested_net_cash = -float(trade_flows["Amount_num"].sum())
    return benchmark_value_series, invested_net_cash


def annualized_return(final_value: float, initial_value: float, years: float) -> float | None:
    if years <= 0 or initial_value <= 0 or final_value <= 0:
        return None
    return (final_value / initial_value) ** (1 / years) - 1


def xnpv(rate: float, cashflows: list[tuple[pd.Timestamp, float]]) -> float:
    if not cashflows:
        return 0.0
    start_date = min(dt for dt, _ in cashflows)
    return sum(amount / ((1 + rate) ** (((dt - start_date).days) / 365.25)) for dt, amount in cashflows)


def xirr(cashflows: list[tuple[pd.Timestamp, float]]) -> float | None:
    if len(cashflows) < 2:
        return None
    amounts = [amount for _, amount in cashflows]
    if not (any(amount < 0 for amount in amounts) and any(amount > 0 for amount in amounts)):
        return None

    low, high = -0.9999, 1.0
    f_low = xnpv(low, cashflows)
    f_high = xnpv(high, cashflows)

    expand_count = 0
    while f_low * f_high > 0 and expand_count < 50:
        high *= 2
        f_high = xnpv(high, cashflows)
        expand_count += 1

    if f_low * f_high > 0:
        return None

    for _ in range(200):
        mid = (low + high) / 2
        f_mid = xnpv(mid, cashflows)
        if abs(f_mid) < 1e-7:
            return mid
        if f_low * f_mid <= 0:
            high = mid
            f_high = f_mid
        else:
            low = mid
            f_low = f_mid

    return (low + high) / 2


def aggregate_cashflows(cashflows: list[tuple[pd.Timestamp, float]]) -> list[tuple[pd.Timestamp, float]]:
    if not cashflows:
        return []
    series = defaultdict(float)
    for dt, amount in cashflows:
        series[pd.Timestamp(dt).normalize()] += float(amount)
    return sorted(series.items(), key=lambda item: item[0])


def clip01(value: float | None) -> float:
    if value is None or pd.isna(value):
        return 0.0
    return max(0.0, min(1.0, float(value)))


def scale_between(value: float | None, low: float, high: float) -> float:
    if value is None or pd.isna(value):
        return 0.0
    if high <= low:
        return 0.0
    return clip01((float(value) - low) / (high - low))


def risk_band(score: float) -> str:
    if score <= 20:
        return "Very Conservative"
    if score <= 40:
        return "Conservative"
    if score <= 60:
        return "Moderate"
    if score <= 80:
        return "Aggressive"
    return "Very Aggressive"


def confidence_band(score: float) -> str:
    if score < 0.35:
        return "Low"
    if score < 0.7:
        return "Medium"
    return "High"


def compute_observed_risk_score(*, stated_risk_score: int, max_position_weight: float, top_5_weight: float, effective_holdings: float | None, annualized_turnover: float | None, median_holding_days: float | None, equity_exposure: float | None, max_drawdown: float | None, downside_capture_ratio: float | None, annualized_volatility: float | None, beta_to_benchmark: float | None, years_of_history: float, closed_lot_count: int) -> dict[str, Any]:
    concentration_components = {
        "single_position_weight": clip01((max_position_weight or 0.0) / 0.22),
        "top_5_weight": clip01((top_5_weight or 0.0) / 0.65),
        "effective_holdings": clip01((10 - (effective_holdings or 10)) / 8),
    }
    market_components = {
        "volatility": clip01((annualized_volatility or 0.0) / 0.32),
        "drawdown": clip01(abs(max_drawdown or 0.0) / 0.40),
        "downside_capture": clip01((downside_capture_ratio or 0.0) / 1.15),
        "beta": clip01((beta_to_benchmark or 0.0) / 1.25),
        "equity_exposure": clip01((equity_exposure or 0.0) / 1.0),
    }
    behavior_components = {
        "turnover": clip01((annualized_turnover or 0.0) / 0.50),
        "short_holding_period": clip01((540 - (median_holding_days or 540)) / 540),
    }

    dimension_scores = {
        "concentration_risk": sum(concentration_components.values()) / len(concentration_components),
        "market_risk": sum(market_components.values()) / len(market_components),
        "behavioral_risk": sum(behavior_components.values()) / len(behavior_components),
    }
    dimension_weights = {
        "concentration_risk": 0.4,
        "market_risk": 0.4,
        "behavioral_risk": 0.2,
    }

    weighted_score = sum(dimension_scores[name] * dimension_weights[name] for name in dimension_scores)
    observed_risk_score = round(weighted_score * 100, 1)
    difference = round(observed_risk_score - stated_risk_score, 1)
    alignment_score = round(max(0.0, 100 - abs(difference) * 2), 1)

    if abs(difference) < 8:
        alignment = "Aligned"
    elif difference >= 8:
        alignment = "Observed portfolio risk is higher than stated risk"
    else:
        alignment = "Observed portfolio risk is lower than stated risk"

    confidence_score = round(min(1.0, (years_of_history / 3.0) * 0.5 + (closed_lot_count / 25.0) * 0.5), 3)

    return {
        "score": observed_risk_score,
        "band": risk_band(observed_risk_score),
        "stated_score": stated_risk_score,
        "stated_band": risk_band(stated_risk_score),
        "difference_vs_stated": difference,
        "alignment": alignment,
        "alignment_score": alignment_score,
        "confidence_score": confidence_score,
        "confidence_band": confidence_band(confidence_score),
        "dimension_scores": {k: round(v * 100, 1) for k, v in dimension_scores.items()},
        "component_scores": {
            **{f"concentration::{k}": round(v * 100, 1) for k, v in concentration_components.items()},
            **{f"market::{k}": round(v * 100, 1) for k, v in market_components.items()},
            **{f"behavior::{k}": round(v * 100, 1) for k, v in behavior_components.items()},
        },
    }


def build_market_enriched_metrics(df: pd.DataFrame, portfolio_summary: dict[str, Any], lot_data: dict[str, Any], market_data: dict[str, Any], benchmark_symbol: str, projection_years: int, stated_risk_score: int) -> dict[str, Any]:
    ticker_close = market_data["ticker_close"]
    latest_prices = market_data["latest_prices"]
    benchmark_close = market_data["benchmark_close"]

    if benchmark_close.empty:
        raise RuntimeError(f"No benchmark history returned for {benchmark_symbol}")

    open_positions_df = pd.DataFrame(lot_data["current_positions"]).copy()
    if open_positions_df.empty:
        open_positions_df = pd.DataFrame(columns=["ticker", "quantity", "cost_basis_total", "avg_cost", "first_buy_date", "weighted_avg_buy_date"])

    open_positions_df["latest_price"] = open_positions_df["ticker"].map(latest_prices)
    open_positions_df["current_value"] = open_positions_df["quantity"] * open_positions_df["latest_price"]
    open_positions_df["unrealized_pnl"] = open_positions_df["current_value"] - open_positions_df["cost_basis_total"]
    open_positions_df["unrealized_return_pct"] = open_positions_df["unrealized_pnl"] / open_positions_df["cost_basis_total"]
    open_positions_df["benchmark_return_since_buy"] = open_positions_df["weighted_avg_buy_date"].apply(lambda d: price_return_from_date(benchmark_close, d) if pd.notna(d) else None)
    open_positions_df["excess_return_vs_benchmark"] = open_positions_df["unrealized_return_pct"] - open_positions_df["benchmark_return_since_buy"]

    current_portfolio_value = float(open_positions_df["current_value"].fillna(0.0).sum())
    total_unrealized_pnl = float(open_positions_df["unrealized_pnl"].fillna(0.0).sum())
    total_realized_pnl = float(sum(lot_data["realized_pnl_by_ticker"].values()))
    raw_cash_balance_estimate = float(df["Amount_num"].sum())
    uninvested_cash_estimate = max(raw_cash_balance_estimate, 0.0)
    total_account_value_estimate = current_portfolio_value + raw_cash_balance_estimate
    net_deposits = float(df.loc[df["Trans Code"] == "ACH", "Amount_num"].sum())
    total_profit_estimate = total_account_value_estimate - net_deposits
    total_return_estimate = (total_profit_estimate / net_deposits) if net_deposits else None

    invested_net_cash_estimate = -float(df.loc[df["Trans Code"].isin(["Buy", "Sell"]), "Amount_num"].sum())
    invested_only_profit_estimate = current_portfolio_value - invested_net_cash_estimate
    invested_only_return_estimate = (invested_only_profit_estimate / invested_net_cash_estimate) if invested_net_cash_estimate else None

    if current_portfolio_value > 0:
        open_positions_df["current_weight"] = open_positions_df["current_value"] / current_portfolio_value
    else:
        open_positions_df["current_weight"] = 0.0

    hhi = float((open_positions_df["current_weight"].fillna(0.0) ** 2).sum())
    max_position_weight = float(open_positions_df["current_weight"].max()) if not open_positions_df.empty else 0.0
    effective_holdings = (1.0 / hhi) if hhi > 0 else None
    top_5_weight = float(open_positions_df.nlargest(5, "current_value")["current_weight"].sum()) if not open_positions_df.empty else 0.0
    concentration_adjusted_return_proxy = invested_only_return_estimate * (1 - hhi) if invested_only_return_estimate is not None else None

    benchmark_value_series, benchmark_invested_net_cash = build_trade_matched_benchmark_series(df, benchmark_close)
    benchmark_current_value = float(benchmark_value_series.iloc[-1]) if not benchmark_value_series.empty else 0.0
    benchmark_return_estimate = ((benchmark_current_value - benchmark_invested_net_cash) / benchmark_invested_net_cash) if benchmark_invested_net_cash else None
    excess_return_vs_benchmark = (invested_only_return_estimate - benchmark_return_estimate) if invested_only_return_estimate is not None and benchmark_return_estimate is not None else None

    market_index = benchmark_close.index
    tracked_tickers = sorted(ticker_close.columns.tolist()) if not ticker_close.empty else []
    price_matrix = ticker_close.reindex(market_index).ffill().reindex(columns=tracked_tickers)
    share_matrix = build_daily_share_matrix(df, market_index, tracked_tickers)
    cash_balance_series = build_cash_balance_series(df, market_index)
    portfolio_equity_series = (share_matrix * price_matrix).sum(axis=1)
    account_value_series = portfolio_equity_series + cash_balance_series

    active_mask = portfolio_equity_series.ne(0) | benchmark_value_series.ne(0)
    if active_mask.any():
        first_active = active_mask[active_mask].index[0]
        portfolio_equity_series = portfolio_equity_series.loc[first_active:]
        account_value_series = account_value_series.loc[first_active:]
        benchmark_value_series = benchmark_value_series.loc[first_active:]

    portfolio_returns = portfolio_equity_series.pct_change().replace([math.inf, -math.inf], pd.NA).fillna(0.0)
    benchmark_returns = benchmark_value_series.pct_change().replace([math.inf, -math.inf], pd.NA).fillna(0.0)
    aligned_returns = pd.concat([portfolio_returns.rename("portfolio"), benchmark_returns.rename("benchmark")], axis=1).dropna()
    portfolio_drawdown = (portfolio_equity_series / portfolio_equity_series.cummax()) - 1
    benchmark_drawdown = (benchmark_value_series / benchmark_value_series.cummax()) - 1
    annualized_volatility = float(portfolio_returns.std() * math.sqrt(252)) if len(portfolio_returns) > 2 else None
    benchmark_annualized_volatility = float(benchmark_returns.std() * math.sqrt(252)) if len(benchmark_returns) > 2 else None
    beta_to_benchmark = None
    tracking_error = None
    info_ratio_proxy = None
    if len(aligned_returns) > 5 and aligned_returns["benchmark"].var() > 0:
        beta_to_benchmark = float(aligned_returns["portfolio"].cov(aligned_returns["benchmark"]) / aligned_returns["benchmark"].var())
        active_return_series = aligned_returns["portfolio"] - aligned_returns["benchmark"]
        tracking_error = float(active_return_series.std() * math.sqrt(252)) if len(active_return_series) > 2 else None
        if tracking_error and tracking_error > 0:
            info_ratio_proxy = float((active_return_series.mean() * 252) / tracking_error)

    worst_benchmark_date = benchmark_drawdown.idxmin() if not benchmark_drawdown.empty else None
    portfolio_drawdown_on_worst_benchmark_day = float(portfolio_drawdown.loc[worst_benchmark_date]) if worst_benchmark_date is not None and worst_benchmark_date in portfolio_drawdown.index else None
    downside_mask = benchmark_returns < 0
    downside_capture_ratio = None
    if downside_mask.any() and benchmark_returns[downside_mask].mean() != 0:
        downside_capture_ratio = float(portfolio_returns[downside_mask].mean() / benchmark_returns[downside_mask].mean())

    years = float(portfolio_summary["date_range"]["years"])
    portfolio_cagr = annualized_return(total_account_value_estimate, net_deposits, years)
    invested_only_cagr = annualized_return(current_portfolio_value, invested_net_cash_estimate, years)
    benchmark_cagr = annualized_return(benchmark_current_value, benchmark_invested_net_cash, years)
    median_holding_days = portfolio_summary["behavioral_metrics"].get("median_holding_period_days")
    annualized_turnover = portfolio_summary["behavioral_metrics"].get("annualized_turnover_proxy")
    equity_exposure = current_portfolio_value / total_account_value_estimate if total_account_value_estimate > 0 else 0.0

    account_cashflows = aggregate_cashflows([
        (row["Activity Date"], -float(row["Amount_num"]))
        for _, row in df[df["Trans Code"] == "ACH"].iterrows()
    ] + [(pd.Timestamp.today().normalize(), total_account_value_estimate)])
    invested_cashflows = aggregate_cashflows([
        (row["Activity Date"], float(row["Amount_num"]))
        for _, row in df[df["Trans Code"].isin(["Buy", "Sell"])].iterrows()
    ] + [(pd.Timestamp.today().normalize(), current_portfolio_value)])
    benchmark_cashflows = aggregate_cashflows([
        (row["Activity Date"], float(row["Amount_num"]))
        for _, row in df[df["Trans Code"].isin(["Buy", "Sell"])].iterrows()
    ] + [(pd.Timestamp.today().normalize(), benchmark_current_value)])

    account_xirr = xirr(account_cashflows)
    invested_only_xirr = xirr(invested_cashflows)
    benchmark_xirr = xirr(benchmark_cashflows)
    excess_xirr_vs_benchmark = (invested_only_xirr - benchmark_xirr) if invested_only_xirr is not None and benchmark_xirr is not None else None

    risk_score = compute_observed_risk_score(
        stated_risk_score=stated_risk_score,
        max_position_weight=max_position_weight,
        top_5_weight=top_5_weight,
        effective_holdings=effective_holdings,
        annualized_turnover=annualized_turnover,
        median_holding_days=median_holding_days,
        equity_exposure=equity_exposure,
        max_drawdown=float(portfolio_drawdown.min()) if not portfolio_drawdown.empty else None,
        downside_capture_ratio=downside_capture_ratio,
        annualized_volatility=annualized_volatility,
        beta_to_benchmark=beta_to_benchmark,
        years_of_history=years,
        closed_lot_count=len(lot_data["closed_lots"]),
    )

    realized_map = lot_data["realized_pnl_by_ticker"].copy()
    unrealized_map = {row["ticker"]: float(row["unrealized_pnl"]) for _, row in open_positions_df.iterrows() if pd.notna(row["unrealized_pnl"])}
    combined_map = defaultdict(float)
    for ticker, pnl in realized_map.items():
        combined_map[ticker] += float(pnl)
    for ticker, pnl in unrealized_map.items():
        combined_map[ticker] += float(pnl)

    total_combined_pnl = sum(combined_map.values())
    attribution_rows = []
    for ticker, pnl in sorted(combined_map.items(), key=lambda item: item[1], reverse=True):
        attribution_rows.append({
            "ticker": ticker,
            "realized_pnl": round(float(realized_map.get(ticker, 0.0)), 2),
            "unrealized_pnl": round(float(unrealized_map.get(ticker, 0.0)), 2),
            "combined_pnl": round(float(pnl), 2),
            "pnl_contribution_pct": round((float(pnl) / total_combined_pnl) * 100, 2) if total_combined_pnl else None,
        })

    closed_lots_df = pd.DataFrame(lot_data["closed_lots"]).copy()
    sold_too_early = pd.DataFrame(columns=["ticker", "missed_upside", "realized_pnl", "if_held_pnl"])
    selection_alpha = pd.DataFrame(columns=["ticker", "alpha_pnl"])

    if not closed_lots_df.empty:
        closed_lots_df["current_price"] = closed_lots_df["ticker"].map(latest_prices)
        closed_lots_df["if_held_pnl"] = closed_lots_df["quantity"] * (closed_lots_df["current_price"] - closed_lots_df["unit_cost"])
        closed_lots_df["missed_upside"] = closed_lots_df["if_held_pnl"] - closed_lots_df["realized_pnl"]
        closed_lots_df["realized_return_pct"] = closed_lots_df["realized_pnl"] / closed_lots_df["cost_basis"]
        closed_lots_df["benchmark_return_during_hold"] = closed_lots_df.apply(lambda row: price_return_between_dates(benchmark_close, row["buy_date"], row["sell_date"]), axis=1)
        closed_lots_df["excess_return_vs_benchmark"] = closed_lots_df["realized_return_pct"] - closed_lots_df["benchmark_return_during_hold"]
        closed_lots_df["alpha_pnl"] = closed_lots_df["cost_basis"] * closed_lots_df["excess_return_vs_benchmark"]

        sold_too_early = closed_lots_df.groupby("ticker", dropna=False)[["missed_upside", "realized_pnl", "if_held_pnl"]].sum().sort_values("missed_upside", ascending=False).reset_index()
        selection_alpha = closed_lots_df.groupby("ticker", dropna=False)["alpha_pnl"].sum().reset_index()

    open_alpha = open_positions_df[["ticker", "cost_basis_total", "excess_return_vs_benchmark"]].copy()
    if not open_alpha.empty:
        open_alpha["alpha_pnl"] = open_alpha["cost_basis_total"] * open_alpha["excess_return_vs_benchmark"]
        open_alpha = open_alpha[["ticker", "alpha_pnl"]]
        selection_alpha = pd.concat([selection_alpha, open_alpha], ignore_index=True)

    if not selection_alpha.empty:
        selection_alpha = selection_alpha.groupby("ticker", dropna=False)["alpha_pnl"].sum().sort_values(ascending=False).reset_index()

    conservative_rate = benchmark_xirr if benchmark_xirr is not None else benchmark_cagr if benchmark_cagr is not None else 0.05
    base_rate = benchmark_xirr if benchmark_xirr is not None else benchmark_cagr if benchmark_cagr is not None else 0.08
    optimistic_rate = max(invested_only_xirr if invested_only_xirr is not None else invested_only_cagr if invested_only_cagr is not None else 0.1, (benchmark_xirr if benchmark_xirr is not None else benchmark_cagr if benchmark_cagr is not None else 0.08) * 1.2)
    projection_rates = {
        "conservative": max(0.03, conservative_rate * 0.75),
        "base": max(0.05, base_rate),
        "optimistic": max(0.07, optimistic_rate),
    }
    projection_scenarios = {name: round(total_account_value_estimate * ((1 + rate) ** projection_years), 2) for name, rate in projection_rates.items()}

    headline_metrics = {
        "benchmark_symbol": benchmark_symbol,
        "benchmark_method": "Trade-matched S&P 500 comparison excluding uninvested cash, with money-weighted return metrics",
        "current_portfolio_value": round(current_portfolio_value, 2),
        "uninvested_cash_estimate": round(uninvested_cash_estimate, 2),
        "raw_cash_balance_estimate": round(raw_cash_balance_estimate, 2),
        "total_account_value_estimate": round(total_account_value_estimate, 2),
        "invested_net_cash_estimate": round(invested_net_cash_estimate, 2),
        "benchmark_invested_net_cash": round(benchmark_invested_net_cash, 2),
        "benchmark_current_value": round(benchmark_current_value, 2),
        "total_realized_pnl": round(total_realized_pnl, 2),
        "total_unrealized_pnl": round(total_unrealized_pnl, 2),
        "total_profit_estimate": round(total_profit_estimate, 2),
        "total_return_estimate_including_cash": round(total_return_estimate, 4) if total_return_estimate is not None else None,
        "invested_only_profit_estimate": round(invested_only_profit_estimate, 2),
        "invested_only_return_estimate": round(invested_only_return_estimate, 4) if invested_only_return_estimate is not None else None,
        "benchmark_return_estimate": round(benchmark_return_estimate, 4) if benchmark_return_estimate is not None else None,
        "excess_return_vs_benchmark": round(excess_return_vs_benchmark, 4) if excess_return_vs_benchmark is not None else None,
        "account_money_weighted_return": round(account_xirr, 4) if account_xirr is not None else None,
        "invested_only_money_weighted_return": round(invested_only_xirr, 4) if invested_only_xirr is not None else None,
        "benchmark_money_weighted_return": round(benchmark_xirr, 4) if benchmark_xirr is not None else None,
        "excess_money_weighted_return_vs_benchmark": round(excess_xirr_vs_benchmark, 4) if excess_xirr_vs_benchmark is not None else None,
        "portfolio_cagr_including_cash": round(portfolio_cagr, 4) if portfolio_cagr is not None else None,
        "invested_only_cagr": round(invested_only_cagr, 4) if invested_only_cagr is not None else None,
        "benchmark_cagr": round(benchmark_cagr, 4) if benchmark_cagr is not None else None,
        "annualized_volatility": round(annualized_volatility, 4) if annualized_volatility is not None else None,
        "benchmark_annualized_volatility": round(benchmark_annualized_volatility, 4) if benchmark_annualized_volatility is not None else None,
        "beta_to_benchmark": round(beta_to_benchmark, 4) if beta_to_benchmark is not None else None,
        "tracking_error": round(tracking_error, 4) if tracking_error is not None else None,
        "information_ratio_proxy": round(info_ratio_proxy, 4) if info_ratio_proxy is not None else None,
        "concentration_hhi": round(hhi, 4),
        "max_position_weight": round(max_position_weight, 4),
        "effective_holdings": round(effective_holdings, 2) if effective_holdings is not None else None,
        "top_5_weight": round(top_5_weight, 4),
        "concentration_adjusted_return_proxy": round(concentration_adjusted_return_proxy, 4) if concentration_adjusted_return_proxy is not None else None,
        "max_portfolio_drawdown": round(float(portfolio_drawdown.min()), 4) if not portfolio_drawdown.empty else None,
        "max_benchmark_drawdown": round(float(benchmark_drawdown.min()), 4) if not benchmark_drawdown.empty else None,
        "portfolio_drawdown_on_worst_benchmark_day": round(portfolio_drawdown_on_worst_benchmark_day, 4) if portfolio_drawdown_on_worst_benchmark_day is not None else None,
        "downside_capture_ratio": round(downside_capture_ratio, 4) if downside_capture_ratio is not None else None,
    }

    return {
        "headline_metrics": headline_metrics,
        "risk_score": risk_score,
        "open_positions": open_positions_df.sort_values("current_value", ascending=False).round(4).to_dict(orient="records"),
        "performance_attribution": attribution_rows[:15],
        "sold_too_early": sold_too_early.head(15).round(4).to_dict(orient="records") if not sold_too_early.empty else [],
        "selection_alpha": selection_alpha.head(15).round(4).to_dict(orient="records") if not selection_alpha.empty else [],
        "projection_scenarios_no_new_contributions": {
            "years": projection_years,
            "annual_return_assumptions": {k: round(v, 4) for k, v in projection_rates.items()},
            "future_values": projection_scenarios,
        },
    }


In [184]:
def ollama_tags(base_url: str = OLLAMA_BASE_URL) -> dict[str, Any]:
    request = urllib.request.Request(f"{base_url}/api/tags", method="GET")
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def ollama_available(model_name: str, base_url: str = OLLAMA_BASE_URL) -> bool:
    try:
        payload = ollama_tags(base_url)
    except Exception:
        return False
    names = {item.get("name") for item in payload.get("models", [])}
    return model_name in names


def pull_model(model_name: str) -> subprocess.CompletedProcess[str]:
    return subprocess.run(["ollama", "pull", model_name], check=True, text=True, capture_output=True)


try:
    tags = ollama_tags()
    print("Ollama is reachable.")
    print("Available models:")
    print([item.get("name") for item in tags.get("models", [])])
except Exception as exc:
    print("Could not reach Ollama at", OLLAMA_BASE_URL)
    print("Start it in a terminal with: ollama serve")
    print("Error:", exc)


Ollama is reachable.
Available models:
['llama3.3:latest', 'llama3.1:latest', 'gemma:2b']


In [185]:
if not ollama_available(MODEL_NAME):
    print(f"Pulling {MODEL_NAME}...")
    result = pull_model(MODEL_NAME)
    print(result.stdout)
else:
    print(f"{MODEL_NAME} is already available locally.")


llama3.1:latest is already available locally.


In [186]:
def build_messages(analysis_payload: dict[str, Any], risk_profile: int) -> list[dict[str, str]]:
    system_prompt = (
        "You are an investment portfolio analysis assistant. "
        "Use only the supplied structured portfolio data, latest prices, benchmark metrics, and risk scoring outputs. "
        "Do not invent prices, news, or fundamentals. "
        "Do not claim certainty about future returns. "
        "Provide educational portfolio analysis rather than guaranteed financial advice. "
        "Return concise markdown with these sections: "
        "1. Investor Profile "
        "2. Portfolio Snapshot "
        "3. Actual Portfolio Risk Score "
        "4. Performance vs S&P 500 "
        "5. Which Holdings Drove Returns "
        "6. Were Winners Sold Too Early? "
        "7. Important Caveats"
    )

    user_prompt = (
        f"User input:\n"
        f"- Risk profile: {risk_profile}/100\n\n"
        "Analysis payload:\n"
        f"{json.dumps(analysis_payload, indent=2)}\n\n"
        "Analyze this investor and provide:\n"
        "- the actual observed portfolio risk score, confidence level, and what drove it\n"
        "- whether the observed risk is aligned with the stated risk profile\n"
        "- current portfolio value, uninvested cash, and unrealized P&L context\n"
        "- whether the investor outperformed or underperformed the S&P 500 using the trade-matched benchmark that excludes uninvested cash\n"
        "- use the money-weighted return comparison when discussing benchmark outperformance\n"
        "- which holdings truly drove performance\n"
        "- whether the results look like stock selection skill or broad market tailwind\n"
        "- any signs that winners were sold too early\n"
        "- clear caveats about what can and cannot be concluded from this data"
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def call_ollama(model: str, messages: list[dict[str, str]], base_url: str = OLLAMA_BASE_URL, temperature: float = TEMPERATURE, num_predict: int = NUM_PREDICT) -> str:
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(f"{base_url}/api/chat", data=data, headers={"Content-Type": "application/json"}, method="POST")

    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace") if exc.fp else ""
        details = raw
        try:
            parsed = json.loads(raw) if raw else {}
            details = parsed.get("error") or parsed.get("message") or raw
        except Exception:
            pass
        raise RuntimeError(f"Ollama /api/chat failed with HTTP {exc.code}: {details}. If this mentions context length or memory, reduce NUM_PREDICT or shorten the payload.") from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Could not connect to Ollama at {base_url}. Ensure Ollama is running (`ollama serve`).") from exc

    message = body.get("message", {})
    content = message.get("content")
    if not content:
        raise RuntimeError(f"Unexpected Ollama response: {body}")
    return content.strip()


In [187]:
transactions = load_transactions(CSV_PATH)
lot_data = build_lot_analytics(transactions)
portfolio_summary = summarize_portfolio(transactions, lot_data)

{
    "date_range": portfolio_summary["date_range"],
    "transaction_counts": portfolio_summary["transaction_counts"],
    "cash_flows": portfolio_summary["cash_flows"],
}


/var/folders/p1/g_l_6ln97jx73n6xnw93ss2h0000gn/T/ipykernel_10332/1008114767.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Activity Date"] = pd.to_datetime(df["Activity Date"], errors="coerce")


{'date_range': {'start': '2020-08-10', 'end': '2026-03-06', 'years': 5.57},
 'transaction_counts': {'rows': 924,
  'buys': 384,
  'sells': 43,
  'distinct_symbols_traded': 61},
 'cash_flows': {'net_deposits': 196787.98,
  'gross_buys': 120684.08,
  'gross_sells': 15979.57,
  'dividends': 1006.44,
  'interest': 3382.45,
  'cash_balance_estimate': 83553.03}}

In [188]:
traded_symbols = sorted(set(transactions.loc[transactions["Instrument"].ne(""), "Instrument"].tolist()))
market_data = fetch_market_data(traded_symbols=traded_symbols, benchmark_symbol=BENCHMARK_SYMBOL, start_date=portfolio_summary["date_range"]["start"])
market_metrics = build_market_enriched_metrics(
    df=transactions,
    portfolio_summary=portfolio_summary,
    lot_data=lot_data,
    market_data=market_data,
    benchmark_symbol=BENCHMARK_SYMBOL,
    projection_years=PROJECTION_YEARS,
    stated_risk_score=RISK_PROFILE,
)
analysis_payload = {"portfolio_summary": portfolio_summary, "market_metrics": market_metrics}
market_metrics["headline_metrics"]


$BRK.B: possibly delisted; no timezone found
$BRK.A: possibly delisted; no timezone found

2 Failed downloads:
['BRK.B', 'BRK.A']: possibly delisted; no timezone found
/var/folders/p1/g_l_6ln97jx73n6xnw93ss2h0000gn/T/ipykernel_10332/4109214202.py:292: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  portfolio_returns = portfolio_equity_series.pct_change().replace([math.inf, -math.inf], pd.NA).fillna(0.0)


{'benchmark_symbol': '^GSPC',
 'benchmark_method': 'Trade-matched S&P 500 comparison excluding uninvested cash, with money-weighted return metrics',
 'current_portfolio_value': 134355.12,
 'uninvested_cash_estimate': 83553.03,
 'raw_cash_balance_estimate': 83553.03,
 'total_account_value_estimate': 217908.15,
 'invested_net_cash_estimate': 104704.51,
 'benchmark_invested_net_cash': 104704.51,
 'benchmark_current_value': 121609.54,
 'total_realized_pnl': 1953.16,
 'total_unrealized_pnl': 29644.56,
 'total_profit_estimate': 21120.17,
 'total_return_estimate_including_cash': 0.1073,
 'invested_only_profit_estimate': 29650.61,
 'invested_only_return_estimate': 0.2832,
 'benchmark_return_estimate': 0.1615,
 'excess_return_vs_benchmark': 0.1217,
 'account_money_weighted_return': 0.0801,
 'invested_only_money_weighted_return': 0.1664,
 'benchmark_money_weighted_return': 0.1004,
 'excess_money_weighted_return_vs_benchmark': 0.066,
 'portfolio_cagr_including_cash': 0.0185,
 'invested_only_cagr'

In [189]:
open_positions_with_prices = pd.DataFrame(market_metrics["open_positions"])
performance_attribution = pd.DataFrame(market_metrics["performance_attribution"])
sold_too_early = pd.DataFrame(market_metrics["sold_too_early"])
selection_alpha = pd.DataFrame(market_metrics["selection_alpha"])

open_positions_with_prices.head(15)


,ticker,quantity,cost_basis_total,avg_cost,first_buy_date,weighted_avg_buy_date,latest_price,current_value,unrealized_pnl,unrealized_return_pct,benchmark_return_since_buy,excess_return_vs_benchmark,current_weight
0,NVDA,140.5482,9882.96,70.3172,2021-08-26,2023-05-16,176.0850,24748.4383,14865.4783,1.5042,0.5992,0.9050,0.1842
1,VOO,27.2101,14052.82,516.4563,2022-01-26,2024-08-18,601.9350,16378.7012,2325.8812,0.1655,0.1719,-0.0064,0.1219
2,AAPL,44.1875,9119.43,206.3803,2020-08-11,2024-05-05,254.1200,11228.9273,2109.4973,0.2313,0.2686,-0.0373,0.0836
3,GOOGL,27.1562,4175.63,153.7637,2022-01-26,2024-01-25,296.2100,8043.9236,3868.2936,0.9264,0.3429,0.5835,0.0599
4,MSFT,21.4145,8730.21,407.6777,2021-10-12,2024-05-20,371.5701,7956.9849,-773.2251,-0.0886,0.2382,-0.3268,0.0592
5,QQQ,13.3571,6724.93,503.4706,2024-05-13,2025-01-19,583.6605,7796.0382,1071.1082,0.1593,0.0865,0.0728,0.0580
6,TSM,21.8845,4761.71,217.5836,2024-11-04,2025-04-20,347.0600,7595.2363,2833.5263,0.5951,0.2742,0.3209,0.0565
7,META,12.9549,6915.61,533.8223,2021-07-14,2024-02-28,579.2400,7503.9909,588.3809,0.0851,0.2964,-0.2113,0.0559
8,COST,6.8168,6229.16,913.7936,2024-08-09,2025-02-25,993.7700,6774.3444,545.1844,0.0875,0.1036,-0.0161,0.0504
9,AMZN,27.7975,5710.90,205.4469,2022-07-08,2025-02-06,210.7800,5859.1469,148.2469,0.0260,0.0804,-0.0544,0.0436


In [190]:
performance_attribution.head(15)


,ticker,realized_pnl,unrealized_pnl,combined_pnl,pnl_contribution_pct
0,NVDA,1744.82,14865.48,16610.30,52.57
1,GOOGL,0.00,3868.29,3868.29,12.24
2,TSM,0.00,2833.53,2833.53,8.97
3,VOO,0.00,2325.88,2325.88,7.36
4,AAPL,28.91,2109.50,2138.41,6.77
5,QQQ,0.00,1071.11,1071.11,3.39
6,TSLA,610.46,382.03,992.49,3.14
7,META,0.00,588.38,588.38,1.86
8,EADSY,15.81,545.73,561.54,1.78
9,HOOD,0.00,558.82,558.82,1.77


In [191]:
sold_too_early.head(15)


,ticker,missed_upside,realized_pnl,if_held_pnl
0,MSFT,553.9257,266.21,820.1357
1,AAPL,543.3777,28.91,572.2877
2,AZN,445.9200,-16.47,429.4500
3,AMAT,349.5600,84.92,434.4800
4,BABA,279.4944,-777.00,-497.5056
5,SPY,228.0870,176.77,404.8570
6,IMVT,152.7518,6.84,159.5918
7,LCID,141.0546,-517.00,-375.9454
8,DAL,116.7200,73.84,190.5600
9,UBER,108.0319,92.11,200.1419


In [192]:
print(json.dumps(analysis_payload["market_metrics"]["risk_score"], indent=2))


{
  "score": 51.0,
  "band": "Moderate",
  "stated_score": 65,
  "stated_band": "Aggressive",
  "difference_vs_stated": -14.0,
  "alignment": "Observed portfolio risk is lower than stated risk",
  "alignment_score": 72.0,
  "confidence_score": 1.0,
  "confidence_band": "High",
  "dimension_scores": {
    "concentration_risk": 54.0,
    "market_risk": 72.3,
    "behavioral_risk": 2.4
  },
  "component_scores": {
    "concentration::single_position_weight": 83.7,
    "concentration::top_5_weight": 78.3,
    "concentration::effective_holdings": 0.0,
    "market::volatility": 100.0,
    "market::drawdown": 100.0,
    "market::downside_capture": 0.0,
    "market::beta": 100.0,
    "market::equity_exposure": 61.7,
    "behavior::turnover": 4.8,
    "behavior::short_holding_period": 0.0
  }
}


In [193]:
print(json.dumps(analysis_payload["market_metrics"]["headline_metrics"], indent=2))


{
  "benchmark_symbol": "^GSPC",
  "benchmark_method": "Trade-matched S&P 500 comparison excluding uninvested cash, with money-weighted return metrics",
  "current_portfolio_value": 134355.12,
  "uninvested_cash_estimate": 83553.03,
  "raw_cash_balance_estimate": 83553.03,
  "total_account_value_estimate": 217908.15,
  "invested_net_cash_estimate": 104704.51,
  "benchmark_invested_net_cash": 104704.51,
  "benchmark_current_value": 121609.54,
  "total_realized_pnl": 1953.16,
  "total_unrealized_pnl": 29644.56,
  "total_profit_estimate": 21120.17,
  "total_return_estimate_including_cash": 0.1073,
  "invested_only_profit_estimate": 29650.61,
  "invested_only_return_estimate": 0.2832,
  "benchmark_return_estimate": 0.1615,
  "excess_return_vs_benchmark": 0.1217,
  "account_money_weighted_return": 0.0801,
  "invested_only_money_weighted_return": 0.1664,
  "benchmark_money_weighted_return": 0.1004,
  "excess_money_weighted_return_vs_benchmark": 0.066,
  "portfolio_cagr_including_cash": 0.018

In [194]:
messages = build_messages(analysis_payload=analysis_payload, risk_profile=RISK_PROFILE)
print(messages[1]["content"][:5000])


User input:
- Risk profile: 65/100

Analysis payload:
{
  "portfolio_summary": {
    "date_range": {
      "start": "2020-08-10",
      "end": "2026-03-06",
      "years": 5.57
    },
    "transaction_counts": {
      "rows": 924,
      "buys": 384,
      "sells": 43,
      "distinct_symbols_traded": 61
    },
    "cash_flows": {
      "net_deposits": 196787.98,
      "gross_buys": 120684.08,
      "gross_sells": 15979.57,
      "dividends": 1006.44,
      "interest": 3382.45,
      "cash_balance_estimate": 83553.03
    },
    "behavioral_metrics": {
      "sell_to_buy_ratio": 0.132,
      "annualized_turnover_proxy": 0.024,
      "holding_period_buckets": {
        "31-364d": 32,
        "<=30d": 2,
        ">=365d": 45
      },
      "median_holding_period_days": 690.0
    },
    "current_positions": [
      {
        "ticker": "VOO",
        "quantity": 27.210083,
        "cost_basis_total": 14052.82,
        "avg_cost": 516.456345,
        "first_buy_date": "2022-01-26",
        "w

In [195]:
analysis = call_ollama(model=MODEL_NAME, messages=messages, temperature=TEMPERATURE, num_predict=NUM_PREDICT)
print(analysis)


**Investor Profile**
-------------------

Risk profile: 65/100 (stated)

**Portfolio Snapshot**
---------------------

* Current portfolio value: $134,355.12
* Uninvested cash estimate: $83,553.03
* Unrealized P&L: $29,644.56
* Total account value estimate: $217,908.15

**Actual Portfolio Risk Score**
-----------------------------

The observed risk score is 51 (Moderate), which is lower than the stated risk profile of 65 (Aggressive). The difference between stated and observed risk is -14 points.

The confidence level in this risk assessment is High, indicating that the model has a high degree of certainty in its predictions.

The component scores driving the observed risk are:

* Concentration::single_position_weight: 83.7
* Market::volatility: 100.0
* Market::drawdown: 100.0
* Market::beta: 100.0

These scores indicate that the portfolio's concentration in individual positions, market volatility, and drawdowns are driving its observed risk.

**Performance vs S&P 500**
--------------